# RNN — Predict Tomorrow's Weather
**Dataset:** Fake daily temperature sequence (no download needed)
**Input:** Last 3 days temperatures | **Output:** Hot(1) or Cold(0)

> RNN = Recurrent Neural Network. Has memory — it uses past values to predict future ones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Input

# ── Fake Temperature Data ─────────────────────────────────
temperatures = [
    38,39,40,41,38,  10,11, 9,12,10,
    35,36,37,38,36,   8, 9,11,10, 9,
    40,41,39,38,40,  12,10, 9,11,12,
    37,38,36,39,37,  11,10,12, 9,11,
]

temps = np.array(temperatures, dtype='float32')
temps = (temps - temps.min()) / (temps.max() - temps.min())  # normalize 0-1

# Build sequences: last 3 days -> next day label
X, y = [], []
for i in range(len(temps) - 3):
    X.append(temps[i:i+3])
    y.append(1 if temps[i+3] > 0.5 else 0)  # 1=Hot, 0=Cold

X = np.array(X).reshape(-1, 3, 1)   # (samples, 3 timesteps, 1 feature)
y = np.array(y)

# ── Train / Test Split ─────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
# ── Plot Temperature Sequence ──────────────────────────────
plt.figure(figsize=(10, 3))
plt.plot(temperatures, marker='o', markersize=4, color='tomato')
plt.axhline(y=25, color='royalblue', linestyle='--', label='Hot/Cold threshold (25°C)')
plt.title('Fake Daily Temperature Data')
plt.xlabel('Day'); plt.ylabel('Temperature (°C)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ── Build RNN (2 stacked layers) ─────────────────────────
model = Sequential([
    Input(shape=(3, 1)),                                        # 3 timesteps, 1 feature
    SimpleRNN(16, return_sequences=True),  # layer 1 passes output to next RNN
    SimpleRNN(8),                          # layer 2
    Dense(1, activation='sigmoid')        # Hot or Cold
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────
history = model.fit(X_train, y_train, epochs=80, batch_size=4,
                    validation_data=(X_test, y_test), verbose=0)

print(f'Train Accuracy: {history.history["accuracy"][-1]*100:.1f}%')
print(f'Test  Accuracy: {history.history["val_accuracy"][-1]*100:.1f}%')

In [ ]:
# ── Accuracy Plot ─────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'],     label='Train')
plt.plot(history.history['val_accuracy'], label='Test', linestyle='--')
plt.title('RNN — Training Accuracy over Epochs')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ── Predict Tomorrow's Weather ─────────────────────────────
test_cases = [
    {'desc': '3 hot days',  'seq': [0.95, 0.97, 0.93]},
    {'desc': '3 cold days', 'seq': [0.02, 0.05, 0.03]},
    {'desc': 'hot->cold',   'seq': [0.90, 0.20, 0.10]},
]

print('─' * 45)
for t in test_cases:
    inp  = np.array(t['seq']).reshape(1, 3, 1)
    prob = model.predict(inp, verbose=0)[0][0]
    result = 'HOT 🌞' if prob > 0.5 else 'COLD ❄️'
    print(f"{t['desc']:12s} → Tomorrow: {result} ({prob*100:.0f}% Hot)")
print('─' * 45)